In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

# 1. Load cleaned silver dataset
df_clean = spark.table("nyc_mobility.clean.green_taxi")
total_rows = df_clean.count()

# 2. Dynamic IQR Calculation for trip_distance
quantiles = df_clean.stat.approxQuantile("trip_distance", [0.25, 0.75], 0.01)
q1, q3 = quantiles[0], quantiles[1]
iqr = q3 - q1
lower_bound, upper_bound = max(0, q1 - 1.5 * iqr), q3 + 1.5 * iqr

# 3. Standard TLC Lookup Values
valid_vendor_ids = [1, 2, 7]
valid_ratecode_ids = [1, 2, 3, 4, 5, 6, 99]
valid_payment_types = [1, 2, 3, 4, 5, 6]
valid_trip_types = [1, 2]

# 4. Define Rules Array: (Column, Check Type, Expression Condition, Severity Mode)
# Severity Mode: "FAIL" triggers FAILED if > 0, "WARN" triggers WARN if > 0
dq_checks = [
    # --- COMPLETENESS ---

    # Pickup/Dropoff/Locations/Total: Verifies critical operational fields are non-null (isNotNull()).
    ("lpep_pickup_datetime", "Completeness", F.col("lpep_pickup_datetime").isNotNull(), "FAIL"),
    ("lpep_dropoff_datetime", "Completeness", F.col("lpep_dropoff_datetime").isNotNull(), "FAIL"),
    ("PULocationID", "Completeness", F.col("PULocationID").isNotNull(), "FAIL"),
    ("DOLocationID", "Completeness", F.col("DOLocationID").isNotNull(), "FAIL"),
    ("total_amount", "Completeness", F.col("total_amount").isNotNull(), "FAIL"),

    # Flags whether the field is populated; expected to trigger WARN as it is permanently 100% null in TLC datasets.
    ("ehail_fee", "Completeness", F.col("ehail_fee").isNotNull(), "WARN"),

    # Checks that records aren't hitting the Vendor 6 bug pattern where VendorID == 6 coincides with missing RatecodeID metadata.
    ("VendorID_6_Metadata", "Completeness", ~((F.col("VendorID") == 6) & F.col("RatecodeID").isNull()), "WARN"),
    
    # --- VALIDITY ---

    # 264/265 represent unknown/out-of-zone
    ("PULocationID", "Validity", (F.col("PULocationID") >= 1) & (F.col("PULocationID") <= 263), "WARN"), 

    # Validates categorical integer codes against official TLC lookup lists (isin()).
    ("DOLocationID", "Validity", (F.col("DOLocationID") >= 1) & (F.col("DOLocationID") <= 263), "WARN"),
    ("VendorID", "Validity", F.col("VendorID").isin(valid_vendor_ids) | F.col("VendorID").isNull(), "WARN"), # Vendor 6 flags as WARN

    ("RatecodeID", "Validity", F.col("RatecodeID").isin(valid_ratecode_ids) | F.col("RatecodeID").isNull(), "FAIL"),
    ("payment_type", "Validity", F.col("payment_type").isin(valid_payment_types) | F.col("payment_type").isNull(), "FAIL"),
    ("trip_type", "Validity", F.col("trip_type").isin(valid_trip_types) | F.col("trip_type").isNull(), "FAIL"),

    # Verifies passenger count is greater than zero, flagging empty or zero-passenger trips.
    ("passenger_count", "Validity", F.col("passenger_count") > 0, "WARN"),
    
    # --- TIMELINESS ---

    # Strictly verifies pickups fall within the expected target dataset period (March 1, 2026 – May 31, 2026).
    ("lpep_pickup_datetime", "Timeliness", (F.col("lpep_pickup_datetime") >= "2026-03-01") & (F.col("lpep_pickup_datetime") < "2026-06-01"), "FAIL"),

    # Checks that trips do not predate March 1, 2026, catching rollover/pre-2026 encoding errors while letting valid late-Feb cross-month trips raise a WARN
    ("lpep_pickup_datetime", "Timeliness", F.col("lpep_pickup_datetime") >= "2026-03-01", "WARN"),
    
    # --- CONSISTENCY ---

    # Ensures chronological order where dropoff happens at or after pickup time.
    ("lpep_dropoff_datetime", "Consistency", F.col("lpep_dropoff_datetime") >= F.col("lpep_pickup_datetime"), "FAIL"),

    # Flags impossible trips moving distance in 0 minutes.
    ("trip_duration_min", "Consistency", ~((F.col("trip_duration_min") == 0) & (F.col("trip_distance") > 0)), "WARN"),

    # Flags instant cancellation/no-show fee charges.
    ("trip_duration_min", "Consistency", ~((F.col("trip_duration_min") == 0) & (F.col("trip_distance") == 0) & (F.col("fare_amount") > 0)), "WARN"),

    # Flags unusually long trips (>3 hours/180 mins) indicating driver-forgotten meters.
    ("trip_duration_min", "Consistency", F.col("trip_duration_min") <= 180, "WARN"),
    
    # --- ACCURACY ---

    # Checks for non-negative financial values (>= 0), catching driver adjustments, voids, or negative balance errors.
    ("fare_amount", "Accuracy", F.col("fare_amount") >= 0, "WARN"),
    ("total_amount", "Accuracy", F.col("total_amount") >= 0, "WARN"),
    ("tip_amount", "Accuracy", F.col("tip_amount") >= 0, "WARN"),

    # Catches severe encoding errors (e.g., raw decimal shifts reaching 100k+ miles).
    ("trip_distance", "Accuracy", F.col("trip_distance") <= 100, "WARN"),

    # Dynamically evaluates whether the distance falls within statistical lower and upper limits ($Q1 - 1.5 \times IQR$ to $Q3 + 1.5 \times IQR$)
    ("trip_distance", "Accuracy", (F.col("trip_distance") >= lower_bound) & (F.col("trip_distance") <= upper_bound), "WARN")
]

# 5. Run single-pass aggregation
agg_exprs = [
    F.sum(F.when(~cond, 1).otherwise(0)).alias(f"check_{i}")
    for i, (_, _, cond, _) in enumerate(dq_checks)
]

agg_results = df_clean.agg(*agg_exprs).collect()[0].asDict()

# 6. Format check outputs with WARN logic
results_data = []
for i, (col, check, _, mode) in enumerate(dq_checks):
    failed_rows = agg_results[f"check_{i}"]
    percentage = round((failed_rows / total_rows) * 100, 2) if total_rows > 0 else 0.0
    
    if failed_rows == 0:
        status = "PASS"
    elif mode == "WARN":
        status = "WARN"
    else:
        status = "FAIL"
        
    results_data.append((col, check, failed_rows, total_rows, percentage, status))

# 7. Add Uniqueness Check
duplicate_count = total_rows - df_clean.dropDuplicates().count()
dup_percentage = round((duplicate_count / total_rows) * 100, 2) if total_rows > 0 else 0.0
dup_status = "PASSED" if duplicate_count == 0 else "FAIL"

results_data.append(("ALL_COLUMNS", "Uniqueness", duplicate_count, total_rows, dup_percentage, dup_status))

# 8. Create Spark DataFrame with snake_case schema for Delta Lake compatibility
schema = StructType([
    StructField("column", StringType(), True),
    StructField("data_quality_check", StringType(), True),
    StructField("failed_rows", LongType(), True),
    StructField("total_rows", LongType(), True),
    StructField("percentage", DoubleType(), True),
    StructField("status", StringType(), True)
])

df_validation = spark.createDataFrame(results_data, schema=schema)

# Display results
display(df_validation)

# 9. Overwrite validation table
df_validation.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("nyc_mobility.validation.green_taxi_validation")